# Pulse SDK Async API Examples

This notebook demonstrates comprehensive usage of the Pulse SDK's async/await functionality. The async API provides full control over job flows and enables efficient concurrent processing in async applications.

## Prerequisites

Before running this notebook, ensure you have:
1. Valid Pulse API credentials (set `PULSE_CLIENT_ID` and `PULSE_CLIENT_SECRET` environment variables)
2. The Pulse SDK installed with async dependencies

Let's start by installing the SDK and importing required modules.

In [ ]:
# Install the Pulse SDK if not already installed
!pip install pulse-sdk

In [ ]:
import asyncio
import os
import time
from typing import List, Dict, Any

# Core async components
from pulse.core.async_client import AsyncCoreClient
from pulse.core.async_jobs import AsyncJob
from pulse.core.models import (
    EmbeddingsRequest,
    SentimentRequest,
    ThemesRequest,
    SimilarityRequest,
    ClusteringRequest,
    ExtractionsRequest,
    SummariesRequest,
)

# High-level async components
from pulse.analysis.async_analyzer import AsyncAnalyzer
from pulse.analysis.processes import ThemeGeneration, SentimentProcess, ThemeAllocation
from pulse.async_dsl import AsyncWorkflow
from pulse.async_starters import (
    generate_themes_async,
    sentiment_analysis_async,
    theme_allocation_async,
    compare_similarity_async,
    cluster_analysis_async,
    extract_elements_async,
    summarize_async,
    estimate_usage_async,
)

# Concurrent processing utilities
from pulse.core.async_concurrent import (
    gather_jobs,
    submit_and_gather_jobs,
    wait_for_any_job,
    AsyncJobManager,
    AsyncConnectionPoolManager,
)

print("✅ All imports successful!")

## Sample Data

Let's define some sample data that we'll use throughout the examples.

In [ ]:
# Sample restaurant reviews for analysis
restaurant_reviews = [
    "The food was absolutely delicious and the service was outstanding!",
    "The restaurant was too noisy and the wait time was excessive.",
    "Great atmosphere and friendly staff, will definitely come back.",
    "The prices are reasonable and the portions are generous.",
    "Poor customer service and the food was cold when it arrived.",
    "Excellent wine selection and knowledgeable sommelier.",
    "The dessert was amazing but the main course was disappointing.",
    "Perfect for a romantic dinner, beautiful ambiance.",
    "Fast service and fresh ingredients, highly recommend.",
    "Overpriced for the quality, won't be returning.",
]

# Sample product reviews
product_reviews = [
    "High quality product, exceeded my expectations.",
    "Difficult to use and poor instructions provided.",
    "Great value for the price, very satisfied.",
    "Product arrived damaged, poor packaging.",
    "Excellent customer support and fast shipping.",
]

print(f"📊 Loaded {len(restaurant_reviews)} restaurant reviews")
print(f"📊 Loaded {len(product_reviews)} product reviews")

## 1. AsyncCoreClient - Low-Level Async API

The `AsyncCoreClient` provides direct access to the Pulse API with full async/await support. It's perfect when you need fine-grained control over API calls and job management.

In [ ]:
async def demonstrate_async_core_client():
    """Demonstrate basic AsyncCoreClient usage."""

    # Create client with automatic authentication
    async with AsyncCoreClient.with_client_credentials_async() as client:
        print("🚀 Created AsyncCoreClient with async context manager")

        # Example 1: Create embeddings with automatic job waiting
        print("\n1. Creating embeddings (automatic job waiting)...")
        request = EmbeddingsRequest(inputs=restaurant_reviews[:3], fast=True)

        start_time = time.time()
        embeddings_response = await client.create_embeddings(request)
        duration = time.time() - start_time

        print(
            f"   ✅ Got {len(embeddings_response.embeddings)} embeddings in {duration:.2f}s"
        )
        print(f"   📊 Usage: {embeddings_response.usage.total} credits")

        # Example 2: Manual job control
        print("\n2. Manual job control...")
        sentiment_request = SentimentRequest(
            inputs=restaurant_reviews[:2],
            fast=False,  # Use slower, more accurate processing
        )

        # Submit job without waiting
        job = await client.analyze_sentiment(sentiment_request, await_job_result=False)
        print(f"   📝 Submitted job: {job.id} (status: {job.status})")

        # Manual polling with custom timeout
        print("   ⏳ Waiting for job completion...")
        result = await job.wait(timeout=60.0)

        print(f"   ✅ Job completed! Got {len(result.results)} sentiment results")
        for i, sentiment_result in enumerate(result.results):
            print(
                f"      Text {i+1}: {sentiment_result.sentiment} ({sentiment_result.confidence:.2f})"
            )


# Run the demonstration
await demonstrate_async_core_client()

## 2. Concurrent Processing

One of the key benefits of async support is the ability to process multiple operations concurrently. Let's explore various concurrent processing patterns.

In [ ]:
async def demonstrate_concurrent_processing():
    """Demonstrate concurrent job processing."""

    async with AsyncCoreClient.with_client_credentials_async() as client:
        print("🔄 Concurrent Processing Examples\n")

        # Example 1: Submit multiple jobs concurrently
        print("1. Submitting multiple embedding jobs...")

        # Create requests for different text batches
        batch_requests = [
            EmbeddingsRequest(inputs=restaurant_reviews[i : i + 2], fast=True)
            for i in range(0, 6, 2)  # 3 batches of 2 texts each
        ]

        # Submit all jobs without waiting
        jobs = []
        for i, request in enumerate(batch_requests):
            job = await client.create_embeddings(request, await_job_result=False)
            jobs.append(job)
            print(f"   📤 Submitted batch {i+1}: {job.id}")

        # Gather results with controlled concurrency
        print("\n   ⏳ Gathering results (max 2 concurrent)...")
        start_time = time.time()

        results = await gather_jobs(jobs, max_concurrent=2)

        duration = time.time() - start_time
        print(f"   ✅ Completed {len(results)} jobs in {duration:.2f}s")

        # Example 2: Mixed operation types concurrently
        print("\n2. Running different operations concurrently...")

        sample_texts = restaurant_reviews[:3]

        # Create different types of requests
        embedding_task = client.create_embeddings(
            EmbeddingsRequest(inputs=sample_texts, fast=True)
        )

        sentiment_task = client.analyze_sentiment(
            SentimentRequest(inputs=sample_texts, fast=True)
        )

        themes_task = client.generate_themes(
            ThemesRequest(inputs=sample_texts, min_themes=2, max_themes=3, fast=True)
        )

        # Run all operations concurrently
        start_time = time.time()
        embeddings, sentiments, themes = await asyncio.gather(
            embedding_task, sentiment_task, themes_task
        )
        duration = time.time() - start_time

        print(f"   ✅ Completed 3 different operations in {duration:.2f}s")
        print(f"      - Embeddings: {len(embeddings.embeddings)} vectors")
        print(f"      - Sentiments: {len(sentiments.results)} analyses")
        print(f"      - Themes: {len(themes.themes)} generated")


# Run concurrent processing demo
await demonstrate_concurrent_processing()

## 3. Async Starter Functions

Async starter functions provide the same simple interface as their sync counterparts but with full async/await support.

In [ ]:
async def demonstrate_async_starters():
    """Demonstrate async starter functions."""

    print("🎯 Async Starter Functions Examples\n")

    sample_texts = restaurant_reviews[:4]

    # Example 1: Sequential async operations
    print("1. Sequential async operations...")

    # Generate themes first
    themes_result = await generate_themes_async(
        sample_texts, min_themes=2, max_themes=4, fast=True
    )
    print(f"   🎨 Generated {len(themes_result.themes)} themes:")
    for theme in themes_result.themes:
        print(f"      - {theme.shortLabel}: {theme.description}")

    # Analyze sentiment
    sentiment_result = await sentiment_analysis_async(sample_texts, fast=True)
    print(f"\n   😊 Analyzed sentiment for {len(sentiment_result.results)} texts:")
    for i, result in enumerate(sentiment_result.results):
        print(f"      Text {i+1}: {result.sentiment} ({result.confidence:.2f})")

    # Allocate texts to themes
    theme_labels = [theme.shortLabel for theme in themes_result.themes]
    allocation_result = await theme_allocation_async(
        sample_texts, themes=theme_labels, fast=True
    )
    print(f"\n   🏷️  Theme allocation:")
    for i, (text, theme) in enumerate(
        zip(allocation_result.texts, allocation_result.allocated_themes)
    ):
        print(f"      '{text[:40]}...' → {theme}")

    # Example 2: Concurrent async starters
    print("\n2. Concurrent async starter operations...")

    # Run multiple operations concurrently
    start_time = time.time()

    similarity_task = compare_similarity_async(sample_texts, flatten=True, fast=True)
    cluster_task = cluster_analysis_async(
        sample_texts, k=2, algorithm="kmeans", fast=True
    )
    usage_task = estimate_usage_async("sentiment", sample_texts)

    similarity_result, cluster_result, usage_result = await asyncio.gather(
        similarity_task, cluster_task, usage_task
    )

    duration = time.time() - start_time
    print(f"   ✅ Completed 3 operations concurrently in {duration:.2f}s")

    # Display results
    sim_shape = (
        f"{len(similarity_result.similarity)}x{len(similarity_result.similarity[0])}"
    )
    print(f"      - Similarity matrix: {sim_shape}")
    print(f"      - Clusters: {len(set(cluster_result.labels))} groups")
    print(f"      - Estimated usage: {usage_result.total} credits")


# Run async starters demo
await demonstrate_async_starters()

## 4. AsyncAnalyzer - High-Level Workflows

The `AsyncAnalyzer` provides async support for complex, multi-step analysis workflows with caching and dependency management.

In [ ]:
async def demonstrate_async_analyzer():
    """Demonstrate AsyncAnalyzer workflows."""

    print("🔬 AsyncAnalyzer Workflow Examples\n")

    # Example 1: Basic async workflow
    print("1. Basic async analysis workflow...")

    processes = [
        ThemeGeneration(min_themes=2, max_themes=4),
        SentimentProcess(),
        ThemeAllocation(),  # Depends on ThemeGeneration
    ]

    async with AsyncAnalyzer(
        dataset=restaurant_reviews[:6],
        processes=processes,
        fast=True,
        use_cache=False,  # Disable for demo
    ) as analyzer:
        print("   🚀 Running async analysis pipeline...")

        start_time = time.time()
        results = await analyzer.run()
        duration = time.time() - start_time

        print(f"   ✅ Analysis completed in {duration:.2f}s")

        # Display results
        if hasattr(results, "theme_generation"):
            themes = results.theme_generation
            print(f"\n   🎨 Generated {len(themes.themes)} themes:")
            for theme in themes.themes:
                print(f"      - {theme.shortLabel}: {theme.description}")

        if hasattr(results, "sentiment"):
            sentiment = results.sentiment
            sentiments = [r.sentiment for r in sentiment.results]
            sentiment_counts = {s: sentiments.count(s) for s in set(sentiments)}
            print(f"\n   😊 Sentiment distribution: {sentiment_counts}")

        if hasattr(results, "theme_allocation"):
            allocation = results.theme_allocation
            print(
                f"\n   🏷️  Theme allocation: {len(allocation.assignments)} texts assigned"
            )

    # Example 2: Concurrent analyzers
    print("\n2. Running multiple analyzers concurrently...")

    async def analyze_dataset(name: str, data: List[str]) -> Dict[str, Any]:
        """Analyze a dataset and return summary."""
        processes = [ThemeGeneration(min_themes=1, max_themes=3), SentimentProcess()]

        async with AsyncAnalyzer(
            dataset=data, processes=processes, fast=True, use_cache=False
        ) as analyzer:
            results = await analyzer.run()

            return {
                "name": name,
                "theme_count": len(results.theme_generation.themes),
                "sentiment_distribution": {
                    s: [r.sentiment for r in results.sentiment.results].count(s)
                    for s in set(r.sentiment for r in results.sentiment.results)
                },
            }

    # Run multiple analyzers concurrently
    start_time = time.time()

    tasks = [
        analyze_dataset("Restaurant Reviews", restaurant_reviews[:4]),
        analyze_dataset("Product Reviews", product_reviews),
    ]

    results = await asyncio.gather(*tasks)
    duration = time.time() - start_time

    print(f"   ✅ Analyzed {len(results)} datasets concurrently in {duration:.2f}s")

    for result in results:
        print(f"\n   📊 {result['name']}:")
        print(f"      - Themes: {result['theme_count']}")
        print(f"      - Sentiments: {result['sentiment_distribution']}")


# Run AsyncAnalyzer demo
await demonstrate_async_analyzer()

## 5. AsyncWorkflow DSL

The `AsyncWorkflow` DSL provides a declarative way to build and execute complex analysis pipelines asynchronously.

In [ ]:
async def demonstrate_async_workflow():
    """Demonstrate AsyncWorkflow DSL."""

    print("🏗️  AsyncWorkflow DSL Examples\n")

    # Example 1: Basic workflow pipeline
    print("1. Basic async workflow pipeline...")

    async with AsyncWorkflow() as workflow:
        # Register data source
        workflow.source("reviews", restaurant_reviews[:5])

        # Build pipeline using method chaining
        workflow.theme_generation(
            source="reviews", min_themes=2, max_themes=4, fast=True
        ).sentiment(source="reviews", fast=True).theme_allocation(inputs="reviews")

        print("   🚀 Executing async workflow...")
        start_time = time.time()

        results = await workflow.run()

        duration = time.time() - start_time
        print(f"   ✅ Workflow completed in {duration:.2f}s")

        # Access results
        themes = [theme.shortLabel for theme in results.theme_generation.themes]
        sentiments = [r.sentiment for r in results.sentiment.sentiments]

        print(f"   🎨 Generated themes: {themes}")
        print(
            f"   😊 Sentiment distribution: {dict((s, sentiments.count(s)) for s in set(sentiments))}"
        )

    # Example 2: Advanced workflow with monitoring
    print("\n2. Advanced workflow with async monitoring...")

    # Define async monitoring callbacks
    async def on_process_start(process_id: str):
        print(f"   ⚡ Starting: {process_id}")

    async def on_process_end(process_id: str, result):
        print(f"   ✅ Completed: {process_id}")

    async with AsyncWorkflow() as workflow:
        # Register multiple data sources
        workflow.source("texts", product_reviews)
        workflow.source("custom_themes", ["Quality", "Usability", "Value", "Support"])

        # Add monitoring
        workflow.monitor(
            on_process_start=on_process_start, on_process_end=on_process_end
        )

        # Build complex pipeline
        workflow.sentiment(source="texts", name="product_sentiment").theme_allocation(
            inputs="texts", themes_from="custom_themes", name="theme_assignment"
        ).similarity(source="texts", flatten=True, name="text_similarity")

        print("   🚀 Executing monitored workflow...")
        results = await workflow.run(fast=True)

        print(f"   📊 Results summary:")
        print(
            f"      - Sentiments: {len(results.product_sentiment.sentiments)} analyzed"
        )
        print(
            f"      - Theme assignments: {len(results.theme_assignment.assignments)} made"
        )
        sim_matrix = results.text_similarity.similarity
        print(f"      - Similarity matrix: {len(sim_matrix)}x{len(sim_matrix[0])}")

    # Example 3: Concurrent workflows
    print("\n3. Running multiple workflows concurrently...")

    async def create_analysis_workflow(name: str, data: List[str]):
        """Create and run an analysis workflow."""
        async with AsyncWorkflow() as workflow:
            workflow.source("data", data)
            workflow.theme_generation(source="data", fast=True).sentiment(
                source="data", fast=True
            )

            results = await workflow.run()
            return {
                "name": name,
                "themes": len(results.theme_generation.themes),
                "sentiments": len(results.sentiment.sentiments),
            }

    # Run multiple workflows concurrently
    start_time = time.time()

    workflow_tasks = [
        create_analysis_workflow("Restaurant", restaurant_reviews[:3]),
        create_analysis_workflow("Product", product_reviews[:3]),
    ]

    workflow_results = await asyncio.gather(*workflow_tasks)
    duration = time.time() - start_time

    print(f"   ✅ Completed {len(workflow_results)} workflows in {duration:.2f}s")
    for result in workflow_results:
        print(
            f"      - {result['name']}: {result['themes']} themes, {result['sentiments']} sentiments"
        )


# Run AsyncWorkflow demo
await demonstrate_async_workflow()

## 6. Advanced Async Patterns

Let's explore some advanced async patterns including error handling, timeouts, and resource management.

In [ ]:
async def demonstrate_advanced_patterns():
    """Demonstrate advanced async patterns."""

    print("🔧 Advanced Async Patterns\n")

    # Example 1: Timeout handling
    print("1. Timeout handling...")

    async with AsyncCoreClient.with_client_credentials_async() as client:
        try:
            # Set a very short timeout to demonstrate timeout handling
            request = EmbeddingsRequest(inputs=restaurant_reviews[:2], fast=True)

            # Use asyncio.wait_for for timeout control
            result = await asyncio.wait_for(
                client.create_embeddings(request), timeout=30.0  # 30 second timeout
            )
            print(
                f"   ✅ Operation completed within timeout: {len(result.embeddings)} embeddings"
            )

        except asyncio.TimeoutError:
            print("   ⏰ Operation timed out")
        except Exception as e:
            print(f"   ❌ Error: {e}")

    # Example 2: Error handling with concurrent operations
    print("\n2. Error handling with concurrent operations...")

    async def safe_operation(operation_name: str, coro):
        """Safely execute an async operation with error handling."""
        try:
            result = await coro
            return {"name": operation_name, "success": True, "result": result}
        except Exception as e:
            return {"name": operation_name, "success": False, "error": str(e)}

    async with AsyncCoreClient.with_client_credentials_async() as client:
        # Create a mix of valid and potentially problematic operations
        operations = [
            safe_operation(
                "embeddings",
                client.create_embeddings(
                    EmbeddingsRequest(inputs=restaurant_reviews[:2], fast=True)
                ),
            ),
            safe_operation(
                "sentiment",
                client.analyze_sentiment(
                    SentimentRequest(inputs=product_reviews[:2], fast=True)
                ),
            ),
            safe_operation(
                "themes",
                client.generate_themes(
                    ThemesRequest(
                        inputs=restaurant_reviews[:3],
                        min_themes=1,
                        max_themes=2,
                        fast=True,
                    )
                ),
            ),
        ]

        # Execute all operations concurrently
        results = await asyncio.gather(*operations)

        # Analyze results
        successful = [r for r in results if r["success"]]
        failed = [r for r in results if not r["success"]]

        print(f"   ✅ Successful operations: {len(successful)}")
        print(f"   ❌ Failed operations: {len(failed)}")

        for result in successful:
            print(f"      - {result['name']}: Success")

        for result in failed:
            print(f"      - {result['name']}: {result['error']}")

    # Example 3: Resource management and cleanup
    print("\n3. Resource management and cleanup...")

    class AsyncResourceManager:
        """Example of proper async resource management."""

        def __init__(self):
            self.client = None
            self.active_jobs = []

        async def __aenter__(self):
            self.client = AsyncCoreClient.with_client_credentials_async()
            await self.client.__aenter__()
            print("   🚀 Resources initialized")
            return self

        async def __aexit__(self, exc_type, exc_val, exc_tb):
            # Cancel any active jobs
            if self.active_jobs:
                print(f"   🛑 Cancelling {len(self.active_jobs)} active jobs...")
                for job in self.active_jobs:
                    try:
                        await job.cancel()
                    except:
                        pass  # Ignore cancellation errors

            # Close client
            if self.client:
                await self.client.__aexit__(exc_type, exc_val, exc_tb)
                print("   🔒 Resources cleaned up")

        async def submit_job(self, request):
            """Submit a job and track it for cleanup."""
            job = await self.client.create_embeddings(request, await_job_result=False)
            self.active_jobs.append(job)
            return job

    # Demonstrate resource management
    async with AsyncResourceManager() as manager:
        # Submit some jobs
        request = EmbeddingsRequest(inputs=restaurant_reviews[:2], fast=True)
        job = await manager.submit_job(request)
        print(f"   📝 Submitted job: {job.id}")

        # Wait for completion
        result = await job.wait()
        print(f"   ✅ Job completed: {len(result.embeddings)} embeddings")

        # Remove completed job from tracking
        manager.active_jobs.remove(job)

    print("   🎯 Resource management demo completed")


# Run advanced patterns demo
await demonstrate_advanced_patterns()

## 7. Performance Comparison: Sync vs Async

Let's compare the performance benefits of async processing over sequential processing.

In [ ]:
async def performance_comparison():
    """Compare sync vs async performance."""

    print("⚡ Performance Comparison: Sequential vs Concurrent\n")

    # Test data
    test_batches = [
        restaurant_reviews[i : i + 2] for i in range(0, 6, 2)  # 3 batches of 2 texts
    ]

    async with AsyncCoreClient.with_client_credentials_async() as client:

        # Sequential processing
        print("1. Sequential processing...")
        start_time = time.time()

        sequential_results = []
        for i, batch in enumerate(test_batches):
            request = EmbeddingsRequest(inputs=batch, fast=True)
            result = await client.create_embeddings(request)
            sequential_results.append(result)
            print(f"   ✅ Completed batch {i+1}/{len(test_batches)}")

        sequential_time = time.time() - start_time
        print(f"   ⏱️  Sequential time: {sequential_time:.2f}s")

        # Concurrent processing
        print("\n2. Concurrent processing...")
        start_time = time.time()

        # Submit all jobs
        jobs = []
        for batch in test_batches:
            request = EmbeddingsRequest(inputs=batch, fast=True)
            job = await client.create_embeddings(request, await_job_result=False)
            jobs.append(job)

        print(f"   📤 Submitted {len(jobs)} jobs")

        # Gather results with controlled concurrency
        concurrent_results = await gather_jobs(jobs, max_concurrent=2)

        concurrent_time = time.time() - start_time
        print(f"   ⏱️  Concurrent time: {concurrent_time:.2f}s")

        # Performance analysis
        print("\n📊 Performance Analysis:")
        speedup = sequential_time / concurrent_time if concurrent_time > 0 else 0
        time_saved = sequential_time - concurrent_time

        print(f"   Sequential: {sequential_time:.2f}s")
        print(f"   Concurrent: {concurrent_time:.2f}s")
        print(f"   Speedup: {speedup:.2f}x")
        print(
            f"   Time saved: {time_saved:.2f}s ({time_saved/sequential_time*100:.1f}%)"
        )

        # Verify results are equivalent
        sequential_embeddings = sum(len(r.embeddings) for r in sequential_results)
        concurrent_embeddings = sum(len(r.embeddings) for r in concurrent_results if r)

        print(f"\n✅ Results verification:")
        print(f"   Sequential embeddings: {sequential_embeddings}")
        print(f"   Concurrent embeddings: {concurrent_embeddings}")
        print(f"   Results match: {sequential_embeddings == concurrent_embeddings}")


# Run performance comparison
await performance_comparison()

## 8. Best Practices Summary

Here are the key best practices for using the Pulse SDK's async functionality:

In [ ]:
def print_best_practices():
    """Display async best practices."""

    print("🎯 Async Best Practices\n")

    practices = [
        {
            "title": "1. Always use async context managers",
            "description": "Use `async with` for proper resource cleanup",
            "example": "async with AsyncCoreClient.with_client_credentials_async() as client:",
        },
        {
            "title": "2. Control concurrency levels",
            "description": "Use max_concurrent parameter to avoid overwhelming the API",
            "example": "await gather_jobs(jobs, max_concurrent=3)",
        },
        {
            "title": "3. Handle timeouts appropriately",
            "description": "Use asyncio.wait_for() for operation timeouts",
            "example": "await asyncio.wait_for(operation, timeout=60.0)",
        },
        {
            "title": "4. Use manual job control when needed",
            "description": "Set await_job_result=False for manual job management",
            "example": "job = await client.create_embeddings(request, await_job_result=False)",
        },
        {
            "title": "5. Implement proper error handling",
            "description": "Use try/except blocks and return_exceptions=True in gather()",
            "example": "results = await asyncio.gather(*tasks, return_exceptions=True)",
        },
        {
            "title": "6. Choose the right abstraction level",
            "description": "Use AsyncCoreClient for control, AsyncAnalyzer for workflows, starters for simplicity",
            "example": "await sentiment_analysis_async(texts)  # Simple case",
        },
        {
            "title": "7. Leverage caching in AsyncAnalyzer",
            "description": "Enable caching for repeated analysis workflows",
            "example": 'AsyncAnalyzer(dataset, processes, use_cache=True, cache_dir="./cache")',
        },
        {
            "title": "8. Monitor long-running operations",
            "description": "Use callbacks and monitoring for visibility",
            "example": "await job.wait_with_callback(status_callback)",
        },
    ]

    for practice in practices:
        print(f"✅ {practice['title']}")
        print(f"   {practice['description']}")
        print(f"   Example: {practice['example']}")
        print()

    print("🚀 Migration Tips:")
    migration_tips = [
        "Replace `CoreClient` with `AsyncCoreClient.with_client_credentials_async()`",
        "Add `await` keywords before all API calls",
        "Use `async with` instead of `with` for context managers",
        "Replace `Analyzer` with `AsyncAnalyzer` for workflows",
        "Use `_async` suffix starter functions (e.g., `sentiment_analysis_async`)",
        "Replace `Workflow` with `AsyncWorkflow` for DSL patterns",
        "Consider concurrent processing with `asyncio.gather()` for performance",
    ]

    for i, tip in enumerate(migration_tips, 1):
        print(f"   {i}. {tip}")


# Display best practices
print_best_practices()

## Conclusion

This notebook has demonstrated the comprehensive async/await support in the Pulse SDK. Key takeaways:

### 🎯 **When to Use Async**
- **Web Applications**: FastAPI, Django async views, Flask with async support
- **Concurrent Processing**: Multiple analysis tasks, batch processing
- **Long-Running Operations**: When you need manual control over job polling
- **Resource Efficiency**: Better resource utilization in I/O-bound applications

### 🔧 **Component Overview**
- **AsyncCoreClient**: Low-level API access with full async support
- **AsyncAnalyzer**: High-level workflows with caching and dependency management
- **AsyncWorkflow**: Declarative DSL for building analysis pipelines
- **Async Starters**: Simple one-line functions with async support
- **Concurrent Utilities**: Tools for managing multiple jobs efficiently

### ⚡ **Performance Benefits**
- **Concurrent Processing**: Significant speedup for multiple operations
- **Non-Blocking**: Doesn't block the event loop in async applications
- **Resource Efficiency**: Better connection pooling and resource management
- **Scalability**: Handles high-concurrency scenarios effectively

### 🛡️ **Best Practices Recap**
1. Always use async context managers for resource cleanup
2. Control concurrency levels to avoid API rate limits
3. Implement proper timeout and error handling
4. Choose the right abstraction level for your use case
5. Leverage caching for repeated operations
6. Monitor long-running operations with callbacks

The async API maintains the same intuitive interface as the sync version while providing the flexibility and performance benefits of async/await patterns. Whether you're building a simple script or a complex web application, the Pulse SDK's async support has you covered!